In [1]:
import pandas as pd
import sys
sys.path.append('..')

from data.emission_factors import BIOMETHANE

print("Biomethane lifecycle analysis by feedstock")
print(f"Feedstocks available: {list(BIOMETHANE.keys())}")

Biomethane lifecycle analysis by feedstock
Feedstocks available: ['agricultural_waste', 'food_waste', 'sewage_sludge', 'energy_crops', 'landfill_gas']


In [5]:
# ── BIOCHEMICAL METHANE POTENTIAL DATA ───────────────────────────────────────
# BMP = maximum methane yield under anaerobic digestion conditions.
# Unit: Nm3 CH4 per tonne of volatile solids (VS) in the feedstock.
# These are well-established literature values used in biogas engineering.
# APESA uses similar data for their 300+ annual BMP measurements.
#
# Volatile solids (VS) = the organic fraction of the feedstock that
# can actually be broken down by bacteria. The rest is ash, water, etc.
# A feedstock with high VS content has more material available for digestion.

BMP_DATA = {
    'agricultural_waste': {
        'bmp_nm3_per_tVS':  280,   # Nm3 CH4 per tonne volatile solids
        'VS_content_pct':    85,   # % of dry matter that is volatile solids
        'moisture_pct':      70,   # % water content in raw feedstock
        'transport_km':      20,   # typical collection radius in km
        'ef_kgCO2_per_kWh': 0.023, # lifecycle carbon intensity (from BIOMETHANE)
    },
    'food_waste': {
        'bmp_nm3_per_tVS':  450,
        'VS_content_pct':    90,
        'moisture_pct':      80,
        'transport_km':      15,
        'ef_kgCO2_per_kWh': 0.018,
    },
    'sewage_sludge': {
        'bmp_nm3_per_tVS':  220,
        'VS_content_pct':    70,
        'moisture_pct':      96,
        'transport_km':       5,   # already on-site at wastewater treatment plant
        'ef_kgCO2_per_kWh': 0.028,
    },
    'energy_crops': {
        'bmp_nm3_per_tVS':  340,
        'VS_content_pct':    93,
        'moisture_pct':      15,
        'transport_km':      30,
        'ef_kgCO2_per_kWh': 0.045,
    },
    'landfill_gas': {
        'bmp_nm3_per_tVS':  None,  # landfill gas is collected directly,
        'VS_content_pct':   None,  # not digested — BMP concept does not apply
        'moisture_pct':     None,
        'transport_km':       0,   # captured on-site
        'ef_kgCO2_per_kWh': 0.012,
    },
}

print("BMP data loaded for feedstocks:")
for name, data in BMP_DATA.items():
    bmp = data['bmp_nm3_per_tVS']
    vs  = data['VS_content_pct']
    if bmp:
        print(f"  {name:<25} BMP: {bmp} Nm3/tVS   VS: {vs}%")
    else:
        print(f"  {name:<25} Direct collection — BMP not applicable")

BMP data loaded for feedstocks:
  agricultural_waste        BMP: 280 Nm3/tVS   VS: 85%
  food_waste                BMP: 450 Nm3/tVS   VS: 90%
  sewage_sludge             BMP: 220 Nm3/tVS   VS: 70%
  energy_crops              BMP: 340 Nm3/tVS   VS: 93%
  landfill_gas              Direct collection — BMP not applicable


In [7]:
def methane_yield_per_tonne(feedstock_data):
    """
    Calculate actual methane yield per tonne of RAW feedstock.
    
    BMP is measured per tonne of volatile solids (VS), not per tonne
    of raw feedstock. We need to convert using the VS content percentage.
    
    Steps:
    1. vs_fraction = VS_content_pct / 100
    2. yield_nm3 = BMP × vs_fraction  (Nm3 CH4 per tonne raw feedstock)
    3. yield_mwh = yield_nm3 × 0.01   (1 Nm3 CH4 = 0.01 MWh, LHV basis)
    """
    if feedstock_data['bmp_nm3_per_tVS'] is None:
        return None   # landfill gas — not applicable

    vs_fraction = feedstock_data['VS_content_pct'] / 100
    bmp         = feedstock_data['bmp_nm3_per_tVS']

    yield_nm3   = bmp * vs_fraction        # Nm3 CH4 per tonne raw feedstock
    yield_mwh   = yield_nm3 * 0.01         # MWh per tonne raw feedstock

    return round(yield_mwh, 3)

# Calculate and display yield for each feedstock
print(f"{'Feedstock':<22} {'Yield (MWh/t)':<15} {'EF (kgCO2/kWh)':<17} {'Carbon per MWh'}")
print('-' * 70)

for name, data in BMP_DATA.items():
    yield_mwh = methane_yield_per_tonne(data)
    ef        = data['ef_kgCO2_per_kWh']

    if yield_mwh is not None:
        carbon_per_mwh = ef * 1000   # convert to gCO2/kWh for readability
        print(f"{name:<22} {yield_mwh:<15.3f} {ef:<17.3f} {carbon_per_mwh:.0f} gCO2/kWh")
    else:
        print(f"{name:<22} {'Direct collection':<15} {ef:<17.3f} {ef*1000:.0f} gCO2/kWh")
    

Feedstock              Yield (MWh/t)   EF (kgCO2/kWh)    Carbon per MWh
----------------------------------------------------------------------
agricultural_waste     2.380           0.023             23 gCO2/kWh
food_waste             4.050           0.018             18 gCO2/kWh
sewage_sludge          1.540           0.028             28 gCO2/kWh
energy_crops           3.162           0.045             45 gCO2/kWh
landfill_gas           Direct collection 0.012             12 gCO2/kWh


In [9]:
# Build a clean summary DataFrame for reporting
rows = []
for name, data in BMP_DATA.items():
    yield_mwh = methane_yield_per_tonne(data)
    rows.append({
        'Feedstock':          name.replace('_', ' ').title(),
        'BMP (Nm3/tVS)':      data['bmp_nm3_per_tVS'] if data['bmp_nm3_per_tVS'] else 'N/A',
        'VS content (%)':     data['VS_content_pct'] if data['VS_content_pct'] else 'N/A',
        'Yield (MWh/t)':      yield_mwh if yield_mwh else 'Direct',
        'EF (kgCO2/kWh)':     data['ef_kgCO2_per_kWh'],
        'Transport (km)':     data['transport_km'],
    })

df_bmp = pd.DataFrame(rows)
print(df_bmp.to_string(index=False))
print()
print("Best feedstock by energy yield:   ", 
      df_bmp.loc[df_bmp['Yield (MWh/t)'] != 'Direct', 
      'Feedstock'].iloc[df_bmp.loc[df_bmp['Yield (MWh/t)'] != 'Direct', 
      'Yield (MWh/t)'].astype(float).idxmax()])
print("Best feedstock by carbon intensity:", 
      df_bmp.loc[df_bmp['EF (kgCO2/kWh)'].idxmin(), 'Feedstock'])

         Feedstock BMP (Nm3/tVS) VS content (%) Yield (MWh/t)  EF (kgCO2/kWh)  Transport (km)
Agricultural Waste           280             85          2.38           0.023              20
        Food Waste           450             90          4.05           0.018              15
     Sewage Sludge           220             70          1.54           0.028               5
      Energy Crops           340             93         3.162           0.045              30
      Landfill Gas           N/A            N/A        Direct           0.012               0

Best feedstock by energy yield:    Food Waste
Best feedstock by carbon intensity: Landfill Gas
